# 11.3 — Bellman Equations

Bellman equations make long horizons local: instead of trying to reason about an entire future at once, they rewrite value as **one immediate reward plus discounted value of what remains**. In this lesson, you will build tiny Markov decision processes from scratch, inspect every backup numerically, and see why expectation, maximization, bootstrapping, and fixed-point iteration are the core mechanics behind reinforcement learning.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build Bellman equations one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so the backup is never a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, transition tables, and exact Bellman arithmetic.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any random choices.

### 1. Discounted return: reward now plus smaller future rewards

The return is the total consequence of a trajectory. A reward received two steps from now is multiplied by $\gamma^2$, so the discount factor $\gamma$ says how much future utility is worth relative to immediate utility. With $0\le\gamma<1$, very distant rewards matter less and infinite sums can stay finite.

In [ ]:
rewards_w = np.array([1.0, 0.0, 2.0])  # three rewards along one trajectory.
gamma_w = 0.9  # future rewards keep 90% of their value per time step.
powers_w = gamma_w ** np.arange(len(rewards_w))  # [1, gamma, gamma^2].
print("discount powers:", np.round(powers_w, 3))
print("discounted terms:", np.round(powers_w * rewards_w, 3))

▶ What you'll see: the delayed reward 2 is counted as `0.9²·2 = 1.62`, not as the full 2.

In [ ]:
G_w = float(np.sum(powers_w * rewards_w))  # discounted return G = sum_t gamma^t r_t.
print("discounted return G:", round(G_w, 3))
assert round(G_w, 3) == 2.620  # lesson number from 1 + 0 + 1.62.

plt.figure(figsize=(4.4, 3))
plt.bar(["r0", "γ r1", "γ² r2"], powers_w * rewards_w, color="teal")
plt.title("1: discounted pieces of a return")
plt.ylabel("discounted reward")
plt.show()

▶ What you'll see: the first and third bars add to 2.62; the middle reward contributes nothing.

*Why it's done this way:* Bellman equations need a finite, comparable target for consequences. Discounting keeps later rewards meaningful but bounded, so choosing a delayed payoff can still be rational without making every future reward as urgent as the present.

### 2. Bellman expectation: average over actions and next states

For a fixed policy $\pi$, the value of a state is an expectation: average over the action the policy takes, then average over the next state the environment produces. The Bellman expectation equation is

$$V^\pi(s)=\sum_a\pi(a\mid s)\sum_{s'}P(s'\mid s,a)\left(R(s,a,s')+\gamma V^\pi(s')\right).$$

In [ ]:
P_w = np.array([[[0.8, 0.2], [0.1, 0.9]],  # from state 0: action 0/1 transition probabilities.
                [[0.0, 1.0], [0.6, 0.4]]]) # from state 1.
R_w = np.array([[[1.0, 0.0], [0.0, 2.0]],  # reward for s,a,s'.
                [[0.0, 3.0], [1.0, 0.0]]])
pi_w = np.array([[0.7, 0.3], [0.4, 0.6]])  # policy probabilities per state.
V_guess_w = np.array([0.5, 1.0])  # current value estimate.
print("P shape:", P_w.shape, "R shape:", R_w.shape, "pi shape:", pi_w.shape)

▶ What you'll see: transition and reward tensors have shape `(states, actions, next_states)`.

In [ ]:
s_w = 0  # evaluate state 0.
a_values_w = []
for a_w in range(2):
    terms_w = P_w[s_w, a_w] * (R_w[s_w, a_w] + gamma_w * V_guess_w)  # next-state weighted terms.
    a_values_w.append(np.sum(terms_w))
    print(f"action {a_w} terms:", np.round(terms_w, 3), "sum:", round(float(np.sum(terms_w)), 3))
a_values_w = np.array(a_values_w)

▶ What you'll see: each action value is itself an expectation over possible next states.

In [ ]:
backup_s0_w = float(np.sum(pi_w[s_w] * a_values_w))  # policy-weighted average over actions.
print("action values:", np.round(a_values_w, 3))
print("Bellman expectation backup V(s0):", round(backup_s0_w, 3))
assert round(backup_s0_w, 3) == 1.734

plt.figure(figsize=(4.4, 3))
plt.bar(["a0", "a1"], a_values_w, color="steelblue")
plt.title("2: action values before policy averaging")
plt.ylabel("one-step reward + γ future")
plt.show()

▶ What you'll see: the policy blends the two action bars using probabilities `[0.7, 0.3]`.

*Why it's done this way:* a policy value is not the best possible future; it is the average future under the policy you actually follow. The double sum makes both sources of randomness explicit: your randomized action choice and the environment's random transition.

### 3. Bellman optimality: choose the best one-step consequence

If we are not merely evaluating a fixed policy, we replace the policy average with a maximum. The optimal value satisfies

$$V^*(s)=\max_a\sum_{s'}P(s'\mid s,a)\left(R(s,a,s')+\gamma V^*(s')\right).$$

This is the same one-step lookahead, but the agent chooses the action with the largest backed-up consequence.

In [ ]:
V_for_control_w = np.array([0.5, 1.0])  # current future-value estimate used for planning.
q_s0_w = []
for a_w in range(2):
    q_w = float(np.sum(P_w[0, a_w] * (R_w[0, a_w] + gamma_w * V_for_control_w)))
    q_s0_w.append(q_w)
    print(f"Q(s0,a{a_w}) from one-step lookahead:", round(q_w, 3))
q_s0_w = np.array(q_s0_w)

▶ What you'll see: action 1 has a larger one-step backed-up value than action 0 in this toy state.

In [ ]:
best_action_w = int(np.argmax(q_s0_w))
optimal_backup_w = float(np.max(q_s0_w))
print("best action:", best_action_w)
print("Bellman optimality backup V*(s0):", round(optimal_backup_w, 3))
assert best_action_w == 1 and round(optimal_backup_w, 3) == 2.655

plt.figure(figsize=(4.4, 3))
plt.bar(["a0", "a1"], q_s0_w, color=["gray", "seagreen"])
plt.title("3: max replaces policy averaging")
plt.ylabel("candidate Q value")
plt.show()

▶ What you'll see: the green bar is the maximizing action used by the optimality backup.

*Why it's done this way:* control asks what the agent should do, so the backup must compare choices. The max is local, but because each candidate includes $\gamma V(s')$, choosing the largest one-step backup also accounts for downstream consequences.

### 4. Fixed-point iteration: repeated backups settle on a value function

A Bellman equation is a fixed-point equation: the correct value function is unchanged by the Bellman backup. Starting from a rough guess, value iteration repeatedly applies the backup until the numbers stop moving. Discounting makes the backup a contraction, so errors shrink geometrically in small finite MDPs.

In [ ]:
def bellman_opt_backup_w(V_w):
    out_w = np.zeros(2)
    for s_w in range(2):
        qs_w = [np.sum(P_w[s_w, a_w] * (R_w[s_w, a_w] + gamma_w * V_w)) for a_w in range(2)]
        out_w[s_w] = np.max(qs_w)
    return out_w

V_iter_w = np.zeros(2)  # intentionally bad starting guess.
history_w = [V_iter_w.copy()]
print("start V:", V_iter_w)

▶ What you'll see: both states begin at zero before any future consequence is propagated.

In [ ]:
for k_w in range(25):
    V_next_w = bellman_opt_backup_w(V_iter_w)
    history_w.append(V_next_w.copy())
    V_iter_w = V_next_w
print("V after 25 backups:", np.round(V_iter_w, 3))
print("last change:", round(float(np.max(np.abs(history_w[-1] - history_w[-2]))), 6))
assert np.allclose(np.round(V_iter_w, 3), np.array([26.528, 27.846]))

▶ What you'll see: the values grow from zero and then move by smaller and smaller amounts.

In [ ]:
history_w = np.array(history_w)
plt.figure(figsize=(4.8, 3))
plt.plot(history_w[:, 0], label="V(s0)")
plt.plot(history_w[:, 1], label="V(s1)")
plt.title("4: Bellman backups approaching a fixed point")
plt.xlabel("iteration")
plt.ylabel("value")
plt.legend()
plt.show()

▶ What you'll see: both curves rise quickly and then flatten, signaling convergence toward the fixed point.

*Why it's done this way:* the equation defines the destination but not the numbers directly. Iteration turns the definition into an algorithm: each backup uses the current future estimate to produce a better estimate, and the fixed point is where another backup changes nothing.

### 5. Bootstrapped TD/Q update: move partway toward a one-step target

When the model is unknown, an agent often sees one sampled transition rather than the full transition table. The one-step target is $y=r+\gamma V(s')$ or $r+\gamma\max_{a'}Q(s',a')$. A learning rate $\alpha$ moves partway toward that target, which reduces variance and avoids letting one sample overwrite the table.

In [ ]:
reward_w = 1.0
next_value_w = 0.8
q_old_w = 0.4
alpha_w = 0.5
target_w = reward_w + gamma_w * next_value_w
print("target y = r + γV(s'):", round(target_w, 3))
assert round(target_w, 3) == 1.720

▶ What you'll see: immediate reward 1 plus discounted next estimate 0.72 gives a target of 1.72.

In [ ]:
q_new_w = q_old_w + alpha_w * (target_w - q_old_w)
print("old Q:", q_old_w)
print("TD error:", round(target_w - q_old_w, 3))
print("new Q:", round(q_new_w, 3))
assert round(q_new_w, 3) == 1.060

plt.figure(figsize=(4.4, 3))
plt.bar(["old Q", "target", "new Q"], [q_old_w, target_w, q_new_w], color=["gray", "black", "seagreen"])
plt.title("5: partial move toward bootstrap target")
plt.ylabel("value")
plt.show()

▶ What you'll see: the new estimate lands halfway between the old estimate and the target.

*Why it's done this way:* bootstrapping trades bias for speed. The target uses the learner's current estimate of the future, so it can update after one step, but the learning rate is essential because that target may be noisy or wrong.

### 6. Policy weighting and exploration pressure

Values become decisions through a policy. A softmax policy converts logits into action probabilities, so raising a logit shifts probability mass and changes expected consequence. Exploration bonuses then deliberately inflate uncertain actions so the agent can learn what greedy exploitation would hide.

In [ ]:
logits_w = np.array([1.0, 0.0])
exp_w = np.exp(logits_w - np.max(logits_w))
prob_w = exp_w / exp_w.sum()
rewards_now_w = np.array([2.0, 0.0])
expected_reward_w = float(prob_w @ rewards_now_w)
print("softmax probs:", np.round(prob_w, 3))
print("expected immediate reward:", round(expected_reward_w, 3))
assert np.allclose(np.round(prob_w, 3), np.array([0.731, 0.269]))
assert round(expected_reward_w, 3) == 1.462

▶ What you'll see: action 0 receives about 73.1% probability, giving expected reward 1.462.

In [ ]:
mean_w, t_w, count_w, c_w = 0.55, 20, 5, 1.0
bonus_w = c_w * np.sqrt(2 * np.log(t_w) / count_w)
ucb_w = mean_w + bonus_w
print("exploration bonus:", round(bonus_w, 3))
print("UCB index:", round(ucb_w, 3))
assert round(ucb_w, 3) == 1.645

plt.figure(figsize=(4.4, 3))
plt.bar(["mean", "bonus", "UCB"], [mean_w, bonus_w, ucb_w], color=["gray", "orange", "purple"])
plt.title("6: uncertainty raises an action's index")
plt.ylabel("score")
plt.show()

▶ What you'll see: the UCB bar is far above the empirical mean because only five samples have been seen.

*Why it's done this way:* Bellman backups say how to value consequences, but policies decide which consequences get sampled. Softmax makes action choice differentiable and probabilistic; exploration bonuses protect learning from becoming trapped by early, incomplete estimates.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, transition probabilities, Bellman backups, and numerical checks.
import matplotlib.pyplot as plt  # load Matplotlib for the heatmaps, bars, and convergence curves in this lesson.
np.random.seed(0)  # make examples with random sampling reproducible across runs.

## 🟢 Basics (warm-up)

### Basic 1 — Compute a discounted return

**Goal.** Turn a short reward stream into one scalar return, because Bellman targets always combine immediate reward with discounted future consequence. We build it in 2 steps.

In [ ]:
rewards_b1 = np.array([1.0, 0.0, 2.0])  # store a three-step reward sequence.
gamma_b1 = 0.9  # choose the discount factor used throughout the warm-up.
powers_b1 = gamma_b1 ** np.arange(len(rewards_b1))  # compute 1, gamma, gamma squared.
print("powers:", np.round(powers_b1, 3))  # inspect how far-future rewards are scaled.

▶ What you'll see: the discount powers are `[1.0, 0.9, 0.81]`.

In [ ]:
G_b1 = float(np.sum(powers_b1 * rewards_b1))  # sum discounted rewards into the return.
print("return:", round(G_b1, 3))  # inspect the scalar consequence.
assert round(G_b1, 3) == 2.620  # verify the lesson return.
plt.figure(figsize=(4, 3))
plt.bar(["t0", "t1", "t2"], powers_b1 * rewards_b1, color="teal")
plt.title("Basic 1: discounted rewards")
plt.ylabel("discounted contribution")
plt.show()

▶ What you'll see: the delayed reward contributes 1.62 after discounting.

👀 Takeaway: return is not one reward; it is the discounted ledger of future rewards.

### Basic 2 — Separate reward from return

**Goal.** Compare immediate reward with total return, because greedily choosing the largest first reward can miss delayed payoff. We build it in 2 steps.

In [ ]:
path_short_b2 = np.array([2.0, 0.0, 0.0])  # tempting path with high immediate reward.
path_long_b2 = np.array([0.0, 0.0, 4.0])  # patient path with delayed payoff.
gamma_b2 = 0.9  # use the same discount factor.
print("immediate rewards:", path_short_b2[0], path_long_b2[0])  # inspect what a greedy agent sees.

▶ What you'll see: the short path looks better if only the first reward is considered.

In [ ]:
powers_b2 = gamma_b2 ** np.arange(3)  # discount factors for three steps.
returns_b2 = np.array([np.sum(path_short_b2 * powers_b2), np.sum(path_long_b2 * powers_b2)])  # compute both returns.
print("returns:", np.round(returns_b2, 3))  # compare total discounted consequence.
assert np.allclose(np.round(returns_b2, 3), np.array([2.000, 3.240]))
plt.figure(figsize=(4, 3))
plt.bar(["short", "delayed"], returns_b2, color=["gray", "seagreen"])
plt.title("Basic 2: reward vs return")
plt.ylabel("discounted return")
plt.show()

▶ What you'll see: the delayed path wins after discounting even though its first reward is zero.

👀 Takeaway: Bellman reasoning prevents immediate reward from hiding delayed consequences.

### Basic 3 — Shape a tiny MDP table

**Goal.** Build transition and reward arrays with explicit shapes, because Bellman code fails silently when states, actions, and next states are mixed up. We build it in 2 steps.

In [ ]:
P_b3 = np.array([[[0.8, 0.2], [0.1, 0.9]], [[0.0, 1.0], [0.6, 0.4]]])  # P[s,a,s'].
R_b3 = np.array([[[1.0, 0.0], [0.0, 2.0]], [[0.0, 3.0], [1.0, 0.0]]])  # R[s,a,s'].
print("P shape:", P_b3.shape, "R shape:", R_b3.shape)  # inspect table dimensions.
assert P_b3.shape == (2, 2, 2) and R_b3.shape == (2, 2, 2)

▶ What you'll see: both arrays are indexed by state, action, next state.

In [ ]:
row_sums_b3 = P_b3.sum(axis=2)  # each transition distribution over next states should sum to one.
print("transition row sums:\n", row_sums_b3)  # inspect valid probabilities.
plt.figure(figsize=(4, 3))
plt.imshow(P_b3.reshape(4, 2), cmap="Blues", aspect="auto")
plt.colorbar(label="probability")
plt.title("Basic 3: transition rows")
plt.xlabel("next state")
plt.ylabel("state-action row")
plt.show()

▶ What you'll see: every state-action row is a probability distribution over next states.

👀 Takeaway: `P[s,a,s']` and `R[s,a,s']` keep the Bellman sums aligned.

### Basic 4 — One action's expected backup

**Goal.** Compute the inner Bellman sum for one state-action pair, because every value backup starts by averaging possible next states. We build it in 2 steps.

In [ ]:
P_b4 = np.array([0.8, 0.2])  # transition probabilities for one state-action pair.
R_b4 = np.array([1.0, 0.0])  # rewards for landing in each next state.
V_b4 = np.array([0.5, 1.0])  # current future-value estimate for next states.
gamma_b4 = 0.9  # discount for the future value term.
terms_b4 = P_b4 * (R_b4 + gamma_b4 * V_b4)  # probability-weight each next-state consequence.
print("terms:", np.round(terms_b4, 3))

▶ What you'll see: each next state contributes probability times `(reward + discounted future)`.

In [ ]:
q_b4 = float(np.sum(terms_b4))  # expected one-step consequence for this action.
print("Q from inner Bellman sum:", round(q_b4, 3))
assert round(q_b4, 3) == 1.340
plt.figure(figsize=(4, 3))
plt.bar(["s'0", "s'1"], terms_b4, color="steelblue")
plt.title("Basic 4: next-state expectation")
plt.ylabel("weighted consequence")
plt.show()

▶ What you'll see: the common next state dominates the expectation because its probability is 0.8.

👀 Takeaway: an action value is an expectation over what the environment might do next.

### Basic 5 — Policy-weight two actions

**Goal.** Average action values with policy probabilities, because policy evaluation asks what happens under the policy, not under the best action. We build it in 2 steps.

In [ ]:
q_actions_b5 = np.array([1.34, 2.655])  # backed-up values for two actions in one state.
pi_b5 = np.array([0.7, 0.3])  # policy chooses action 0 more often.
weighted_b5 = pi_b5 * q_actions_b5  # action contributions to V^pi.
print("weighted pieces:", np.round(weighted_b5, 3))

▶ What you'll see: action 0 contributes more because it has both high value and high probability.

In [ ]:
v_b5 = float(np.sum(weighted_b5))  # Bellman expectation over actions.
print("policy value:", round(v_b5, 3))
assert round(v_b5, 3) == 1.734
plt.figure(figsize=(4, 3))
plt.bar(["π(a0)Q0", "π(a1)Q1"], weighted_b5, color="purple")
plt.title("Basic 5: policy-weighted action values")
plt.ylabel("contribution")
plt.show()

▶ What you'll see: the two bars add to the state's value under this policy.

👀 Takeaway: policy evaluation is weighted averaging, not maximization.

### Basic 6 — Choose the optimal action

**Goal.** Replace policy averaging with a max, because Bellman optimality backs up the best available action. We build it in 2 steps.

In [ ]:
q_actions_b6 = np.array([1.34, 2.655])  # candidate one-step lookahead values.
best_b6 = int(np.argmax(q_actions_b6))  # identify the maximizing action.
print("candidate Q values:", q_actions_b6)
print("best action:", best_b6)

▶ What you'll see: action 1 has the larger backed-up consequence.

In [ ]:
vstar_b6 = float(np.max(q_actions_b6))  # optimal value backup for the state.
print("optimal backup:", round(vstar_b6, 3))
assert round(vstar_b6, 3) == 2.655
plt.figure(figsize=(4, 3))
plt.bar(["a0", "a1"], q_actions_b6, color=["gray", "seagreen"])
plt.title("Basic 6: max over actions")
plt.ylabel("Q candidate")
plt.show()

▶ What you'll see: the taller bar becomes the backed-up state value.

👀 Takeaway: optimal control changes the Bellman action step from expectation to maximization.

### Basic 7 — One bootstrap target

**Goal.** Build `r + γV(next)` directly, because sampled RL updates often cannot average over the whole transition model. We build it in 2 steps.

In [ ]:
r_b7 = 1.0  # observed immediate reward.
gamma_b7 = 0.9  # discount factor.
next_v_b7 = 0.8  # current estimate of the next state's value.
target_b7 = r_b7 + gamma_b7 * next_v_b7  # one-step bootstrapped target.
print("target:", round(target_b7, 3))
assert round(target_b7, 3) == 1.720

▶ What you'll see: the target combines one real reward with one estimated future value.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["reward", "γ next V", "target"], [r_b7, gamma_b7 * next_v_b7, target_b7], color=["gray", "orange", "teal"])
plt.title("Basic 7: bootstrap target pieces")
plt.ylabel("value")
plt.show()

▶ What you'll see: the target bar is the sum of the reward and discounted future bars.

👀 Takeaway: bootstrapping updates now by borrowing the current estimate of what remains.

### Basic 8 — Move an estimate partway

**Goal.** Apply a learning-rate update, because one noisy target should not completely overwrite an existing estimate. We build it in 2 steps.

In [ ]:
old_b8 = 0.4  # current table estimate.
target_b8 = 1.72  # one-step Bellman target.
alpha_b8 = 0.5  # move halfway toward the target.
td_error_b8 = target_b8 - old_b8  # signed correction signal.
print("TD error:", round(td_error_b8, 3))

▶ What you'll see: the target is 1.32 above the old estimate.

In [ ]:
new_b8 = old_b8 + alpha_b8 * td_error_b8  # incremental Bellman update.
print("new estimate:", round(new_b8, 3))
assert round(new_b8, 3) == 1.060
plt.figure(figsize=(4, 3))
plt.bar(["old", "target", "new"], [old_b8, target_b8, new_b8], color=["gray", "black", "seagreen"])
plt.title("Basic 8: learning-rate interpolation")
plt.ylabel("estimate")
plt.show()

▶ What you'll see: the new estimate lands exactly halfway between old and target.

👀 Takeaway: alpha controls how aggressively a Bellman target changes the table.

### Basic 9 — Softmax action probabilities

**Goal.** Convert logits into a policy distribution, because expected returns need action probabilities that sum to one. We build it in 2 steps.

In [ ]:
logits_b9 = np.array([1.0, 0.0])  # unnormalized action preferences.
exp_b9 = np.exp(logits_b9 - np.max(logits_b9))  # stabilized exponentials.
probs_b9 = exp_b9 / exp_b9.sum()  # normalize to probabilities.
print("probabilities:", np.round(probs_b9, 3))
assert np.allclose(np.round(probs_b9, 3), np.array([0.731, 0.269]))

▶ What you'll see: the larger logit gets more probability but the smaller action still has support.

In [ ]:
rewards_b9 = np.array([2.0, 0.0])  # immediate rewards for each action.
expected_b9 = float(probs_b9 @ rewards_b9)  # policy-weighted expected reward.
print("expected reward:", round(expected_b9, 3))
assert round(expected_b9, 3) == 1.462
plt.figure(figsize=(4, 3))
plt.bar(["a0", "a1"], probs_b9, color="darkorange")
plt.title("Basic 9: softmax policy")
plt.ylabel("probability")
plt.show()

▶ What you'll see: action 0 receives most, but not all, of the probability mass.

👀 Takeaway: policy probabilities are the weights inside expected consequence.

### Basic 10 — Add an exploration bonus

**Goal.** Compute a UCB-style index, because uncertain actions may deserve temporary extra value while the agent learns. We build it in 2 steps.

In [ ]:
mean_b10 = 0.55  # empirical reward mean for one action.
time_b10 = 20  # total decision count.
count_b10 = 5  # times this action was selected.
bonus_b10 = np.sqrt(2 * np.log(time_b10) / count_b10)  # uncertainty bonus.
print("bonus:", round(bonus_b10, 3))

▶ What you'll see: a rarely sampled action receives a large bonus.

In [ ]:
ucb_b10 = mean_b10 + bonus_b10  # optimistic action index.
print("UCB index:", round(ucb_b10, 3))
assert round(ucb_b10, 3) == 1.645
plt.figure(figsize=(4, 3))
plt.bar(["mean", "bonus", "index"], [mean_b10, bonus_b10, ucb_b10], color=["gray", "orange", "purple"])
plt.title("Basic 10: exploration pressure")
plt.ylabel("score")
plt.show()

▶ What you'll see: optimism can make an uncertain action look worth trying.

👀 Takeaway: exploration changes which Bellman targets the learner gets to observe.

## 🟡 Easy

### Easy 1 — Evaluate a fixed policy by repeated Bellman backups

**Goal.** Apply the Bellman expectation backup until it stabilizes, because policy evaluation is solving a fixed-point equation for a chosen policy. We build it in 3 steps.

In [ ]:
P_e1 = np.array([[[0.8, 0.2], [0.1, 0.9]], [[0.0, 1.0], [0.6, 0.4]]])  # transition model P[s,a,s'].
R_e1 = np.array([[[1.0, 0.0], [0.0, 2.0]], [[0.0, 3.0], [1.0, 0.0]]])  # reward model R[s,a,s'].
pi_e1 = np.array([[0.7, 0.3], [0.4, 0.6]])  # fixed policy to evaluate.
gamma_e1 = 0.9  # discount factor.
V_e1 = np.zeros(2)  # start policy values at zero.
print("initial V:", V_e1)

▶ What you'll see: the policy evaluation starts from no future-value knowledge.

In [ ]:
hist_e1 = [V_e1.copy()]  # record the value path for plotting.
for it_e1 in range(40):
    V_new_e1 = np.zeros_like(V_e1)
    for s_e1 in range(2):
        vals_e1 = [np.sum(P_e1[s_e1, a_e1] * (R_e1[s_e1, a_e1] + gamma_e1 * V_e1)) for a_e1 in range(2)]
        V_new_e1[s_e1] = np.sum(pi_e1[s_e1] * vals_e1)
    hist_e1.append(V_new_e1.copy())
    V_e1 = V_new_e1
print("evaluated V:", np.round(V_e1, 3))
assert np.allclose(np.round(V_e1, 3), np.array([12.942, 13.522]))

In [ ]:
hist_e1 = np.array(hist_e1)
plt.figure(figsize=(5, 3))
plt.plot(hist_e1[:, 0], label="Vπ(s0)")
plt.plot(hist_e1[:, 1], label="Vπ(s1)")
plt.title("Easy 1: policy evaluation convergence")
plt.xlabel("backup iteration")
plt.ylabel("value")
plt.legend()
plt.show()

▶ What you'll see: both value curves approach stable policy values.

👀 Takeaway: policy evaluation repeatedly averages one-step consequence under the same policy.

### Easy 2 — Run value iteration for optimal values

**Goal.** Solve for optimal state values, because control uses the max Bellman backup instead of policy weighting. We build it in 3 steps.

In [ ]:
P_e2 = np.array([[[0.8, 0.2], [0.1, 0.9]], [[0.0, 1.0], [0.6, 0.4]]])  # transition model.
R_e2 = np.array([[[1.0, 0.0], [0.0, 2.0]], [[0.0, 3.0], [1.0, 0.0]]])  # reward model.
gamma_e2 = 0.9  # discount factor.
V_e2 = np.zeros(2)  # value iteration starting point.
print("start:", V_e2)

▶ What you'll see: optimal planning starts from a blank value table.

In [ ]:
hist_e2 = [V_e2.copy()]
for it_e2 in range(50):
    V_new_e2 = np.zeros_like(V_e2)
    for s_e2 in range(2):
        qs_e2 = [np.sum(P_e2[s_e2, a_e2] * (R_e2[s_e2, a_e2] + gamma_e2 * V_e2)) for a_e2 in range(2)]
        V_new_e2[s_e2] = np.max(qs_e2)
    hist_e2.append(V_new_e2.copy())
    V_e2 = V_new_e2
print("optimal V:", np.round(V_e2, 3))
assert np.allclose(np.round(V_e2, 3), np.array([28.527, 29.845]))

In [ ]:
hist_e2 = np.array(hist_e2)
plt.figure(figsize=(5, 3))
plt.plot(hist_e2[:, 0], label="V*(s0)")
plt.plot(hist_e2[:, 1], label="V*(s1)")
plt.title("Easy 2: optimal value iteration")
plt.xlabel("backup iteration")
plt.ylabel("value")
plt.legend()
plt.show()

▶ What you'll see: optimal values converge higher than fixed-policy values because the max chooses better actions.

👀 Takeaway: value iteration is repeated Bellman optimality backup until the fixed point is nearly reached.

### Easy 3 — Extract a greedy policy from values

**Goal.** Convert a learned value function into actions, because optimal values are useful only after we choose the maximizing action in each state. We build it in 3 steps.

In [ ]:
P_e3 = np.array([[[0.8, 0.2], [0.1, 0.9]], [[0.0, 1.0], [0.6, 0.4]]])  # transition model.
R_e3 = np.array([[[1.0, 0.0], [0.0, 2.0]], [[0.0, 3.0], [1.0, 0.0]]])  # reward model.
V_e3 = np.array([28.527, 29.845])  # approximate optimal values from value iteration.
gamma_e3 = 0.9  # discount factor.
print("input V:", V_e3)

▶ What you'll see: the value vector summarizes future consequences for both states.

In [ ]:
Q_e3 = np.zeros((2, 2))  # store action values implied by V.
for s_e3 in range(2):
    for a_e3 in range(2):
        Q_e3[s_e3, a_e3] = np.sum(P_e3[s_e3, a_e3] * (R_e3[s_e3, a_e3] + gamma_e3 * V_e3))
policy_e3 = np.argmax(Q_e3, axis=1)  # greedy action per state.
print("Q table:\n", np.round(Q_e3, 3))
print("greedy policy:", policy_e3)
assert np.array_equal(policy_e3, np.array([1, 0]))

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(Q_e3, cmap="viridis", aspect="auto")
plt.colorbar(label="Q value")
plt.title("Easy 3: greedy Q values")
plt.xlabel("action")
plt.ylabel("state")
plt.show()

▶ What you'll see: the brightest cell in each row marks the action selected by the greedy policy.

👀 Takeaway: a greedy policy is the row-wise argmax of one-step lookahead Q values.

### Easy 4 — Perform a sampled Q-learning update

**Goal.** Update one action-value entry from a sampled transition, because model-free control bootstraps from the best next action without needing transition probabilities. We build it in 3 steps.

In [ ]:
Q_e4 = np.array([[0.4, 0.2], [0.8, 0.1]])  # current action-value table.
s_e4, a_e4, r_e4, sp_e4 = 0, 0, 1.0, 1  # sampled transition (state, action, reward, next state).
gamma_e4 = 0.9  # discount factor.
alpha_e4 = 0.5  # learning rate.
print("old Q table:\n", Q_e4)

▶ What you'll see: the update will modify only `Q[0,0]`.

In [ ]:
target_e4 = r_e4 + gamma_e4 * np.max(Q_e4[sp_e4])  # Q-learning target.
td_e4 = target_e4 - Q_e4[s_e4, a_e4]  # difference between target and old estimate.
Q_new_e4 = Q_e4.copy()
Q_new_e4[s_e4, a_e4] += alpha_e4 * td_e4  # move partway toward target.
print("target:", round(target_e4, 3), "TD error:", round(td_e4, 3))
print("new Q[0,0]:", round(Q_new_e4[0, 0], 3))
assert round(target_e4, 3) == 1.720 and round(Q_new_e4[0, 0], 3) == 1.060

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(Q_new_e4, cmap="viridis", aspect="auto")
plt.colorbar(label="Q")
plt.title("Easy 4: Q table after one update")
plt.xlabel("action")
plt.ylabel("state")
plt.show()

▶ What you'll see: one cell becomes brighter after moving from 0.4 to 1.06.

👀 Takeaway: Q-learning is a Bellman optimality backup applied to one sampled table entry.

### Easy 5 — Compare discount factors

**Goal.** Sweep gamma in a delayed-reward problem, because the discount factor controls how much future payoff can outweigh immediate reward. We build it in 3 steps.

In [ ]:
path_now_e5 = np.array([2.0, 0.0, 0.0])  # immediate payoff path.
path_late_e5 = np.array([0.0, 0.0, 4.0])  # delayed payoff path.
gammas_e5 = np.array([0.2, 0.5, 0.8, 0.9, 0.99])  # candidate discount factors.
print("gammas:", gammas_e5)

▶ What you'll see: the sweep spans short-sighted to patient settings.

In [ ]:
returns_now_e5 = []
returns_late_e5 = []
for g_e5 in gammas_e5:
    powers_e5 = g_e5 ** np.arange(3)
    returns_now_e5.append(float(path_now_e5 @ powers_e5))
    returns_late_e5.append(float(path_late_e5 @ powers_e5))
returns_now_e5 = np.array(returns_now_e5)
returns_late_e5 = np.array(returns_late_e5)
print("now returns:", np.round(returns_now_e5, 3))
print("late returns:", np.round(returns_late_e5, 3))
assert returns_late_e5[-1] > returns_now_e5[-1]

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(gammas_e5, returns_now_e5, marker="o", label="immediate")
plt.plot(gammas_e5, returns_late_e5, marker="o", label="delayed")
plt.title("Easy 5: gamma changes patience")
plt.xlabel("gamma")
plt.ylabel("discounted return")
plt.legend()
plt.show()

▶ What you'll see: the delayed path becomes competitive only when gamma is large enough.

👀 Takeaway: gamma is a modeling choice about patience, not a harmless constant.

## 🔴 Advanced

### Advanced 1 — Solve policy evaluation as a linear system

**Goal.** Compare iterative policy evaluation with the exact linear-system solution, because Bellman expectation is a fixed-point equation that can be written as `(I - γPπ)V = rπ`. We build it in 4 steps.

In [ ]:
P_a1 = np.array([[[0.8, 0.2], [0.1, 0.9]], [[0.0, 1.0], [0.6, 0.4]]])  # transition model.
R_a1 = np.array([[[1.0, 0.0], [0.0, 2.0]], [[0.0, 3.0], [1.0, 0.0]]])  # reward model.
pi_a1 = np.array([[0.7, 0.3], [0.4, 0.6]])  # fixed policy.
gamma_a1 = 0.9  # discount factor.
print("policy rows sum:", pi_a1.sum(axis=1))

▶ What you'll see: each policy row is a valid action distribution.

In [ ]:
Ppi_a1 = np.zeros((2, 2))  # policy-induced transition matrix.
rpi_a1 = np.zeros(2)  # policy-induced expected immediate reward.
for s_a1 in range(2):
    for a_a1 in range(2):
        Ppi_a1[s_a1] += pi_a1[s_a1, a_a1] * P_a1[s_a1, a_a1]
        rpi_a1[s_a1] += pi_a1[s_a1, a_a1] * np.sum(P_a1[s_a1, a_a1] * R_a1[s_a1, a_a1])
print("Pπ:\n", np.round(Ppi_a1, 3))
print("rπ:", np.round(rpi_a1, 3))

In [ ]:
V_exact_a1 = np.linalg.solve(np.eye(2) - gamma_a1 * Ppi_a1, rpi_a1)  # exact Bellman fixed point.
print("exact Vπ:", np.round(V_exact_a1, 3))
assert np.allclose(np.round(V_exact_a1, 3), np.array([13.140, 13.721]))

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["s0", "s1"], V_exact_a1, color="navy")
plt.title("Advanced 1: exact policy values")
plt.ylabel("Vπ")
plt.show()

▶ What you'll see: solving the linear system jumps directly to the fixed point.

👀 Takeaway: repeated Bellman expectation backups are iterative solvers for a precise linear equation.

### Advanced 2 — Visualize contraction of two value guesses

**Goal.** Show that Bellman optimality pulls different guesses closer together, because discounting makes the backup a contraction in the max norm. We build it in 4 steps.

In [ ]:
P_a2 = np.array([[[0.8, 0.2], [0.1, 0.9]], [[0.0, 1.0], [0.6, 0.4]]])  # transition model.
R_a2 = np.array([[[1.0, 0.0], [0.0, 2.0]], [[0.0, 3.0], [1.0, 0.0]]])  # reward model.
gamma_a2 = 0.9  # contraction factor upper bound.
V_left_a2 = np.array([0.0, 5.0])  # first rough value guess.
V_right_a2 = np.array([3.0, -1.0])  # second rough value guess.
print("initial max-norm gap:", np.max(np.abs(V_left_a2 - V_right_a2)))

▶ What you'll see: the two value guesses begin far apart.

In [ ]:
def opt_backup_a2(V_a2):
    out_a2 = np.zeros(2)
    for s_a2 in range(2):
        qs_a2 = [np.sum(P_a2[s_a2, a_a2] * (R_a2[s_a2, a_a2] + gamma_a2 * V_a2)) for a_a2 in range(2)]
        out_a2[s_a2] = np.max(qs_a2)
    return out_a2

gaps_a2 = []
for it_a2 in range(12):
    gaps_a2.append(float(np.max(np.abs(V_left_a2 - V_right_a2))))
    V_left_a2 = opt_backup_a2(V_left_a2)
    V_right_a2 = opt_backup_a2(V_right_a2)
print("gaps:", np.round(gaps_a2, 3))
assert gaps_a2[-1] < gaps_a2[0]

In [ ]:
ratios_a2 = np.array(gaps_a2[1:]) / np.array(gaps_a2[:-1])
print("largest observed ratio:", round(float(np.max(ratios_a2)), 3))
assert np.max(ratios_a2) <= gamma_a2 + 1e-9

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(gaps_a2, marker="o", color="crimson")
plt.title("Advanced 2: backup contraction")
plt.xlabel("iteration")
plt.ylabel("max-norm gap")
plt.show()

▶ What you'll see: the gap between guesses shrinks after repeated backups.

👀 Takeaway: the fixed point is reachable because Bellman backups reduce value-function disagreement when gamma is below 1.

### Advanced 3 — Compare full-return Monte Carlo and one-step TD targets

**Goal.** Estimate the same state's value with complete returns and with one-step bootstrapping, because the target choice controls bias and variance. We build it in 4 steps.

In [ ]:
rng_a3 = np.random.default_rng(3)  # local generator for reproducible trajectories.
gamma_a3 = 0.9  # discount factor.
n_a3 = 200  # number of simulated episodes.
returns_a3 = []  # complete Monte Carlo returns.
td_targets_a3 = []  # one-step bootstrap targets.
V_next_est_a3 = 0.8  # deliberately fixed next-state estimate for TD.
print("episodes:", n_a3)

▶ What you'll see: the comparison uses many tiny random episodes.

In [ ]:
for ep_a3 in range(n_a3):
    r0_a3 = rng_a3.choice([0.0, 2.0], p=[0.4, 0.6])
    r1_a3 = rng_a3.choice([0.0, 3.0], p=[0.7, 0.3])
    r2_a3 = rng_a3.choice([0.0, 5.0], p=[0.8, 0.2])
    returns_a3.append(r0_a3 + gamma_a3 * r1_a3 + gamma_a3**2 * r2_a3)
    td_targets_a3.append(r0_a3 + gamma_a3 * V_next_est_a3)
returns_a3 = np.array(returns_a3)
td_targets_a3 = np.array(td_targets_a3)
print("mean MC return:", round(float(np.mean(returns_a3)), 3))
print("mean TD target:", round(float(np.mean(td_targets_a3)), 3))

In [ ]:
std_mc_a3 = float(np.std(returns_a3))
std_td_a3 = float(np.std(td_targets_a3))
print("std MC:", round(std_mc_a3, 3), "std TD:", round(std_td_a3, 3))
assert std_td_a3 < std_mc_a3

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(returns_a3, bins=12, alpha=0.6, label="full return")
plt.hist(td_targets_a3, bins=8, alpha=0.6, label="one-step TD")
plt.title("Advanced 3: target variance")
plt.xlabel("target value")
plt.ylabel("count")
plt.legend()
plt.show()

▶ What you'll see: TD targets are more concentrated because they replace later random rewards with an estimate.

👀 Takeaway: bootstrapping usually lowers variance but can introduce bias through the estimated future value.

### Advanced 4 — Track over-large learning-rate instability

**Goal.** Compare small and large alpha values in repeated TD updates, because bootstrapping from a moving estimate can amplify noise when steps are too aggressive. We build it in 4 steps.

In [ ]:
rng_a4 = np.random.default_rng(4)  # reproducible sampled rewards.
gamma_a4 = 0.9  # discount factor.
alphas_a4 = [0.1, 1.2]  # stable and intentionally aggressive learning rates.
curves_a4 = []  # store value curves.
print("alphas:", alphas_a4)

▶ What you'll see: one alpha is conservative and one overshoots the target repeatedly.

In [ ]:
for alpha_a4 in alphas_a4:
    V_a4 = 0.0
    curve_a4 = []
    for t_a4 in range(60):
        reward_a4 = 1.0 + 0.3 * rng_a4.normal()  # noisy immediate reward.
        target_a4 = reward_a4 + gamma_a4 * V_a4  # bootstrapped target uses current V.
        V_a4 = V_a4 + alpha_a4 * (target_a4 - V_a4)  # TD(0) scalar update.
        curve_a4.append(V_a4)
    curves_a4.append(np.array(curve_a4))
print("final values:", [round(float(c_a4[-1]), 3) for c_a4 in curves_a4])

In [ ]:
stable_range_a4 = float(np.max(curves_a4[0]) - np.min(curves_a4[0]))
aggressive_range_a4 = float(np.max(curves_a4[1]) - np.min(curves_a4[1]))
print("stable range:", round(stable_range_a4, 3), "aggressive range:", round(aggressive_range_a4, 3))
assert aggressive_range_a4 > stable_range_a4

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(curves_a4[0], label="α=0.1")
plt.plot(curves_a4[1], label="α=1.2")
plt.title("Advanced 4: TD step-size sensitivity")
plt.xlabel("update")
plt.ylabel("value estimate")
plt.legend()
plt.show()

▶ What you'll see: the aggressive curve reacts much more sharply to reward noise.

👀 Takeaway: bootstrapped targets move with the estimate, so alpha is a stability knob.

### Advanced 5 — Sweep exploration bonus strength

**Goal.** Simulate UCB-style action selection for several bonus strengths, because exploration pressure changes the data that later Bellman updates can learn from. We build it in 4 steps.

In [ ]:
true_means_a5 = np.array([0.45, 0.55, 0.50])  # hidden reward rates for three actions.
cs_a5 = np.array([0.0, 0.5, 1.5])  # exploration strengths.
steps_a5 = 120  # decisions per run.
print("true means:", true_means_a5)

▶ What you'll see: action 1 is truly best, but action 2 is close enough to confuse early samples.

In [ ]:
counts_all_a5 = []
rewards_all_a5 = []
for c_a5 in cs_a5:
    rng_loop_a5 = np.random.default_rng(int(c_a5 * 10 + 5))
    counts_a5 = np.ones(3)
    sums_a5 = true_means_a5.copy()
    total_reward_a5 = 0.0
    for t_a5 in range(4, steps_a5 + 4):
        means_a5 = sums_a5 / counts_a5
        index_a5 = means_a5 + c_a5 * np.sqrt(2 * np.log(t_a5) / counts_a5)
        action_a5 = int(np.argmax(index_a5))
        reward_a5 = 1.0 if rng_loop_a5.random() < true_means_a5[action_a5] else 0.0
        counts_a5[action_a5] += 1
        sums_a5[action_a5] += reward_a5
        total_reward_a5 += reward_a5
    counts_all_a5.append(counts_a5)
    rewards_all_a5.append(total_reward_a5)
print("final counts:\n", np.round(np.array(counts_all_a5), 1))

In [ ]:
rewards_all_a5 = np.array(rewards_all_a5)
best_c_a5 = float(cs_a5[int(np.argmax(rewards_all_a5))])
print("total rewards:", rewards_all_a5)
print("best c in this run:", best_c_a5)
assert rewards_all_a5.shape == (3,)

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(cs_a5, rewards_all_a5, marker="o", color="purple")
plt.title("Advanced 5: exploration-strength sweep")
plt.xlabel("bonus strength c")
plt.ylabel("total reward over run")
plt.show()

▶ What you'll see: no-bonus exploitation can under-sample alternatives, while too much bonus can waste pulls.

👀 Takeaway: exploration is not separate from Bellman learning; it controls which experience populates the backups.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Bellman equations make long horizons local by asking one step plus what remains.

The Bellman backup turns a long-horizon question into one immediate reward plus discounted downstream value. The residual tells us whether repeated local backups have stabilized. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

SEED = 1106
rng = np.random.default_rng(SEED)
GAMMA = 0.90
ACTIONS = ["up", "right", "down", "left"]
DELTAS = {
    "up": (-1, 0),
    "right": (0, 1),
    "down": (1, 0),
    "left": (0, -1),
}

@dataclass
class GridEnv:
    name: str
    rows: int
    cols: int
    start: int
    states: list
    index_of: dict
    terminal: set
    P: list
    rewards: np.ndarray
    shape_label: str


def discounted_return(rewards, gamma):
    total = 0.0
    power = 1.0
    for reward in rewards:
        total = total + power * reward
        power = power * gamma
    return total


def softmax(logits):
    shifted = np.asarray(logits, dtype=float) - np.max(logits)
    weights = np.exp(shifted)
    return weights / weights.sum()


def move_cell(cell, action, rows, cols, walls):
    dr, dc = DELTAS[action]
    nr = cell[0] + dr
    nc = cell[1] + dc
    candidate = (nr, nc)
    if nr < 0 or nr >= rows:
        return cell
    if nc < 0 or nc >= cols:
        return cell
    if candidate in walls:
        return cell
    return candidate


def build_grid_env(name, rows, cols, start_cell, goal_cells, pit_cells=None, walls=None, step_cost=-0.02, slip=0.0, wind=0.0, bonuses=None):
    pit_cells = set(pit_cells or [])
    walls = set(walls or [])
    bonuses = dict(bonuses or {})
    goal_cells = dict(goal_cells)
    states = []
    for r in range(rows):
        for c in range(cols):
            if (r, c) not in walls:
                states.append((r, c))
    index_of = {cell: i for i, cell in enumerate(states)}
    terminal_cells = set(goal_cells) | pit_cells
    terminal = {index_of[cell] for cell in terminal_cells}
    n_states = len(states)
    rewards = np.zeros(n_states)
    for cell, reward in goal_cells.items():
        rewards[index_of[cell]] = reward
    for cell in pit_cells:
        rewards[index_of[cell]] = -1.0
    for cell, reward in bonuses.items():
        rewards[index_of[cell]] = reward
    P = []
    for state_index, cell in enumerate(states):
        state_rows = []
        for action in ACTIONS:
            if state_index in terminal:
                state_rows.append([(1.0, state_index, 0.0, True)])
                continue
            side_actions = [action, ACTIONS[(ACTIONS.index(action) - 1) % 4], ACTIONS[(ACTIONS.index(action) + 1) % 4]]
            probs = [1.0 - slip, slip / 2.0, slip / 2.0]
            outcomes = {}
            for prob, actual_action in zip(probs, side_actions):
                if prob <= 0.0:
                    continue
                next_cell = move_cell(cell, actual_action, rows, cols, walls)
                windy_cell = move_cell(next_cell, "up", rows, cols, walls)
                wind_options = [(1.0 - wind, next_cell), (wind, windy_cell)]
                for wind_prob, final_cell in wind_options:
                    if wind_prob <= 0.0:
                        continue
                    next_index = index_of[final_cell]
                    done = next_index in terminal
                    reward = step_cost + rewards[next_index]
                    key = (next_index, done, reward)
                    outcomes[key] = outcomes.get(key, 0.0) + prob * wind_prob
            state_rows.append([(prob, ns, rew, done) for (ns, done, rew), prob in outcomes.items()])
        P.append(state_rows)
    shape_label = f"{rows}x{cols}, |S|={n_states}, |A|={len(ACTIONS)}"
    return GridEnv(name, rows, cols, index_of[start_cell], states, index_of, terminal, P, rewards, shape_label)


def two_state_chain():
    return build_grid_env(
        "D1 two-state chain",
        1,
        2,
        (0, 0),
        {(0, 1): 1.0},
        step_cost=0.0,
        slip=0.0,
    )


def build_env_ladder():
    envs = []
    envs.append(two_state_chain())
    envs.append(build_grid_env(
        "D2 slippery 3-state",
        1,
        3,
        (0, 0),
        {(0, 2): 1.0},
        pit_cells={(0, 1)},
        step_cost=-0.01,
        slip=0.20,
    ))
    envs.append(build_grid_env(
        "D3 4x4 gridworld",
        4,
        4,
        (3, 0),
        {(0, 3): 1.0},
        pit_cells={(1, 3)},
        walls={(1, 1), (2, 1)},
        step_cost=-0.03,
        slip=0.05,
    ))
    envs.append(build_grid_env(
        "D4 stochastic windy grid",
        5,
        5,
        (4, 0),
        {(0, 4): 1.2},
        pit_cells={(2, 3), (3, 2)},
        walls={(1, 1), (1, 2), (3, 1)},
        step_cost=-0.04,
        slip=0.15,
        wind=0.20,
    ))
    envs.append(build_grid_env(
        "D5 larger sparse-reward grid",
        8,
        8,
        (7, 0),
        {(0, 7): 2.0},
        pit_cells={(2, 5), (3, 5), (5, 3), (6, 6)},
        walls={(1, 1), (1, 2), (1, 3), (2, 1), (4, 2), (4, 3), (4, 4), (5, 5)},
        step_cost=-0.025,
        slip=0.10,
        wind=0.10,
        bonuses={(7, 1): 0.25},
    ))
    return envs


def q_from_v(env, V, gamma=GAMMA):
    Q = np.zeros((len(env.states), len(ACTIONS)))
    for s in range(len(env.states)):
        for a in range(len(ACTIONS)):
            total = 0.0
            for prob, next_state, reward, done in env.P[s][a]:
                total = total + prob * (reward + gamma * V[next_state] * (not done))
            Q[s, a] = total
    return Q


def policy_evaluation(env, policy, gamma=GAMMA, sweeps=200, tol=1e-10):
    V = np.zeros(len(env.states))
    errors = []
    for sweep in range(sweeps):
        old = V.copy()
        for s in range(len(env.states)):
            if s in env.terminal:
                V[s] = 0.0
                continue
            total = 0.0
            for a in range(len(ACTIONS)):
                for prob, next_state, reward, done in env.P[s][a]:
                    total = total + policy[s, a] * prob * (reward + gamma * old[next_state] * (not done))
            V[s] = total
        errors.append(float(np.max(np.abs(V - old))))
        if errors[-1] < tol:
            break
    return V, np.asarray(errors)


def value_iteration(env, gamma=GAMMA, sweeps=500, tol=1e-10):
    V = np.zeros(len(env.states))
    errors = []
    residuals = []
    for sweep in range(sweeps):
        old = V.copy()
        Q = q_from_v(env, old, gamma)
        for s in range(len(env.states)):
            if s in env.terminal:
                V[s] = 0.0
            else:
                V[s] = np.max(Q[s])
        residual = float(np.max(np.abs(V - old)))
        errors.append(residual)
        residuals.append(residual)
        if residual < tol:
            break
    policy = np.zeros((len(env.states), len(ACTIONS)))
    greedy = np.argmax(q_from_v(env, V, gamma), axis=1)
    for s, action in enumerate(greedy):
        policy[s, action] = 1.0
    return V, policy, np.asarray(errors), np.asarray(residuals)


def policy_iteration(env, gamma=GAMMA, sweeps=80):
    policy = np.ones((len(env.states), len(ACTIONS))) / len(ACTIONS)
    errors = []
    for sweep in range(sweeps):
        V, eval_errors = policy_evaluation(env, policy, gamma=gamma, sweeps=200)
        Q = q_from_v(env, V, gamma)
        greedy = np.argmax(Q, axis=1)
        new_policy = np.zeros_like(policy)
        for s, action in enumerate(greedy):
            new_policy[s, action] = 1.0
        change = float(np.max(np.abs(new_policy - policy)))
        errors.append(change)
        policy = new_policy
        if change == 0.0:
            break
    V, eval_errors = policy_evaluation(env, policy, gamma=gamma, sweeps=300)
    return V, policy, np.asarray(errors)


def run_episode(env, policy, gamma=GAMMA, max_steps=120, epsilon=0.0, start_state=None, rng=None):
    rng = rng or np.random.default_rng(SEED)
    state = env.start if start_state is None else start_state
    trajectory = []
    for step in range(max_steps):
        if state in env.terminal:
            break
        if rng.random() < epsilon:
            action = int(rng.integers(len(ACTIONS)))
        else:
            probs = policy[state] / policy[state].sum()
            action = int(rng.choice(len(ACTIONS), p=probs))
        choices = env.P[state][action]
        probs = np.array([item[0] for item in choices], dtype=float)
        probs = probs / probs.sum()
        choice = int(rng.choice(len(choices), p=probs))
        prob, next_state, reward, done = choices[choice]
        trajectory.append((state, action, reward, next_state, done))
        state = next_state
        if done:
            break
    return trajectory


def episode_return(trajectory, gamma=GAMMA):
    return discounted_return([step[2] for step in trajectory], gamma)


def monte_carlo_value(env, policy, episodes=300, gamma=GAMMA, epsilon=0.10, exploring_starts=False, rng=None):
    rng = rng or np.random.default_rng(SEED)
    V = np.zeros(len(env.states))
    counts = np.zeros(len(env.states))
    errors = []
    V_star, optimal_policy, vi_errors, residuals = value_iteration(env, gamma=gamma)
    for episode in range(episodes):
        start_state = None
        if exploring_starts:
            candidates = [s for s in range(len(env.states)) if s not in env.terminal]
            start_state = int(rng.choice(candidates))
        trajectory = run_episode(env, policy, gamma=gamma, epsilon=epsilon, start_state=start_state, rng=rng)
        G = 0.0
        seen = set()
        for state, action, reward, next_state, done in reversed(trajectory):
            G = reward + gamma * G
            if state in seen:
                continue
            seen.add(state)
            counts[state] = counts[state] + 1.0
            V[state] = V[state] + (G - V[state]) / counts[state]
        errors.append(float(np.max(np.abs(V - V_star))))
    return V, counts, np.asarray(errors)


def td0_value(env, policy, episodes=300, alpha=0.20, gamma=GAMMA, epsilon=0.10, rng=None):
    rng = rng or np.random.default_rng(SEED)
    V = np.zeros(len(env.states))
    errors = []
    V_star, optimal_policy, vi_errors, residuals = value_iteration(env, gamma=gamma)
    for episode in range(episodes):
        trajectory = run_episode(env, policy, gamma=gamma, epsilon=epsilon, rng=rng)
        for state, action, reward, next_state, done in trajectory:
            target = reward + gamma * V[next_state] * (not done)
            V[state] = V[state] + alpha * (target - V[state])
        errors.append(float(np.max(np.abs(V - V_star))))
    return V, np.asarray(errors)


def evaluate_policy_return(env, policy, episodes=80, gamma=GAMMA, rng=None):
    rng = rng or np.random.default_rng(SEED)
    returns = []
    for episode in range(episodes):
        trajectory = run_episode(env, policy, gamma=gamma, rng=rng)
        returns.append(episode_return(trajectory, gamma))
    return float(np.mean(returns))


def immediate_reward_policy(env):
    policy = np.zeros((len(env.states), len(ACTIONS)))
    for s in range(len(env.states)):
        if s in env.terminal:
            policy[s, 0] = 1.0
            continue
        means = []
        for a in range(len(ACTIONS)):
            means.append(sum(prob * reward for prob, next_state, reward, done in env.P[s][a]))
        policy[s, int(np.argmax(means))] = 1.0
    return policy


def uniform_policy(env):
    return np.ones((len(env.states), len(ACTIONS))) / len(ACTIONS)


def value_grid(env, V):
    grid = np.full((env.rows, env.cols), np.nan)
    for idx, cell in enumerate(env.states):
        grid[cell] = V[idx]
    return grid


def policy_grid(env, policy):
    chars = np.full((env.rows, env.cols), " ", dtype=object)
    arrows = np.array(["^", ">", "v", "<"], dtype=object)
    greedy = np.argmax(policy, axis=1)
    for idx, cell in enumerate(env.states):
        chars[cell] = "T" if idx in env.terminal else arrows[greedy[idx]]
    return chars


def plot_value_policy_panels(envs, values, policies, metric_values, metric_name):
    fig, axes = plt.subplots(2, len(envs), figsize=(4 * len(envs), 7))
    for i, env in enumerate(envs):
        ax = axes[0, i]
        image = ax.imshow(value_grid(env, values[i]), cmap="viridis")
        ax.set_title(env.name)
        plt.colorbar(image, ax=ax, fraction=0.046)
        ax = axes[1, i]
        ax.imshow(value_grid(env, values[i]), cmap="viridis")
        arrows = policy_grid(env, policies[i])
        grid = value_grid(env, values[i])
        for r in range(env.rows):
            for c in range(env.cols):
                if not np.isnan(grid[r, c]):
                    ax.text(c, r, arrows[r, c], ha="center", va="center", color="white")
        ax.set_title("greedy policy")
    fig.tight_layout()
    plt.figure(figsize=(7, 3))
    plt.plot(range(1, len(metric_values) + 1), metric_values, marker="o")
    plt.xticks(range(1, len(metric_values) + 1), ["D1", "D2", "D3", "D4", "D5"])
    plt.ylabel(metric_name)
    plt.xlabel("environment rung")
    plt.title(f"{metric_name} across the D1-D5 ladder")
    plt.grid(True, alpha=0.3)
    plt.show()


def print_ladder_preview(envs):
    for env in envs:
        sample = env.states[: min(5, len(env.states))]
        print(f"{env.name}: {env.shape_label}; start={env.states[env.start]}; sample={sample}")


## The concept, built once on D1

We implement a real Bellman optimality backup, then track residuals across deterministic, slippery, windy, and sparse-reward environments.

Formula: $\mathcal{B}V(s)=\max_a\sum_{s'}P(s'\mid s,a)(R(s,a,s')+\gamma V(s'))$

First assert the exact worked numbers from the lesson: discounted return, one-step TD target, softmax policy weighting, and UCB exploration pressure. These are small enough to verify by hand.

In [ ]:
lesson_return = discounted_return([1.0, 0.0, 2.0], 0.9)
td_target = 1.0 + 0.9 * 0.8
q_new = 0.4 + 0.5 * (td_target - 0.4)
policy_probs = softmax([1.0, 0.0])
expected_reward = policy_probs[0] * 2.0 + policy_probs[1] * 0.0
ucb_index = 0.55 + np.sqrt(2.0 * np.log(20.0) / 5.0)
assert np.isclose(lesson_return, 2.620)
assert np.isclose(td_target, 1.720)
assert np.isclose(q_new, 1.060)
assert np.isclose(np.round(policy_probs[0], 3), 0.731)
assert np.isclose(np.round(policy_probs[1], 3), 0.269)
assert np.isclose(np.round(expected_reward, 3), 1.462)
assert np.isclose(np.round(ucb_index, 3), 1.645)
print(lesson_return, td_target, q_new, policy_probs, expected_reward, ucb_index)

A Bellman backup computes expected one-step reward plus discounted future value for every action.

In [ ]:
def bellman_backup(env, V, gamma=GAMMA):
    Q = q_from_v(env, V, gamma)
    backed_up = np.max(Q, axis=1)
    for state in env.terminal:
        backed_up[state] = 0.0
    return backed_up, Q

env = two_state_chain()
V0 = np.zeros(len(env.states))
V1, Q1 = bellman_backup(env, V0)
assert np.isclose(Q1[0, 1], 1.0)
assert np.isclose(V1[0], 1.0)
print(V1)

The lesson's one-step target example also checks: $y=1+0.9\cdot0.8=1.720$ and $Q_{new}=0.4+0.5(1.720-0.4)=1.060$.

In [ ]:
target = 1.0 + 0.9 * 0.8
q_new = 0.4 + 0.5 * (target - 0.4)
assert np.isclose(target, 1.720)
assert np.isclose(q_new, 1.060)
print(target, q_new)

## The dataset ladder

The family F12 ladder is built inline: D1 two-state chain, D2 slippery three-state, D3 4x4 gridworld, D4 stochastic windy grid, and D5 larger sparse-reward grid.

In [ ]:
envs = build_env_ladder()
print_ladder_preview(envs)

## Run the same method across D1-D5

Collect the plan metric: Bellman residual/value-error.

In [ ]:
envs = build_env_ladder()
values = []
policies = []
metrics = []
for env in envs:
    V_star, policy_star, errors, residuals = value_iteration(env)
    final_residual = float(residuals[-1])
    values.append(V_star)
    policies.append(policy_star)
    metrics.append(final_residual)
    print(f"{env.name:28s}  {final_residual: .8f}")

## Results visualization

The closing figure has value/policy heatmap panels for every environment plus one summary curve over D1-D5.

In [ ]:
plot_value_policy_panels(envs, values, policies, metrics, "Bellman residual/value-error")

## Pitfall on the hardest rung

Reproduce the named D5 pitfall, then apply the fix from the lesson.

In [ ]:
env = envs[-1]
V = np.zeros(len(env.states))
unstable = []
for sweep in range(25):
    backed_up, Q = bellman_backup(env, V)
    V = V + 1.40 * (backed_up - V)
    unstable.append(float(np.max(np.abs(backed_up - V))))
V_fixed, policy_fixed, errors, residuals = value_iteration(env)
print(f"overshoot residual: {unstable[-1]:.3f}")
print(f"stable residual: {residuals[-1]:.8f}")
assert residuals[-1] < unstable[-1]

## Evaluate it + Practice

- Metric: Bellman residual/value-error on D1-D5, compared with a no-skill uniform or immediate-reward baseline.
- Sanity check: D1 must match the hand value and the lesson numbers asserted above.
- Ablation: turn off discounted consequence or coverage and verify the metric worsens.
- Failure signal: residuals stop shrinking, value shapes mismatch, or D5 return drops below the baseline.
- Reproducibility: keep the provided seed and do not download simulators.

Practice prompts:
1. Change $\gamma$ from $0.90$ to $0.70$ and predict which rungs lose the most value before running.

2. Add one wall to D3 and inspect how the optimal policy heatmap reroutes around it.

3. On D5, compare the no-skill uniform policy with the learned or planned policy using the same return metric.